# QuickPay Fintech Pipeline
### Parts 3 (Reconciliation) + 4 (JSON Normalization)

In [1]:
import pandas as pd
import numpy as np
import json
import os

DATA_DIR = "../01_data/raw"  # adjust path as needed; expects the 7 input files
OUT_DIR = "../01_data/processed"
os.makedirs(OUT_DIR, exist_ok=True)

# For standalone run, point to uploaded files
import sys
if not os.path.exists(DATA_DIR):
    DATA_DIR = "."  # fallback: files in same dir


## Part 3: Python Reconciliation Workflow
### 3.1 Load Files

In [2]:
ledger = pd.read_csv(f"{DATA_DIR}/ledger.csv")
gateway = pd.read_csv(f"{DATA_DIR}/gateway.csv")

print(f"Ledger: {ledger.shape[0]} rows x {ledger.shape[1]} cols")
print(f"Gateway: {gateway.shape[0]} rows x {gateway.shape[1]} cols")
display(ledger.head())
display(gateway.head())


Ledger: 10 rows x 6 cols
Gateway: 9 rows x 6 cols


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,850.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R004,2026-03-02,M003,2100.0,success,Card
4,R005,2026-03-03,M004,7200.0,success,Card


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,900.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R005,2026-03-03,M004,7200.0,failed,Card
4,R006,2026-03-03,M002,950.0,success,UPI


### 3.2 Check Duplicates and Nulls

In [3]:
print("=== LEDGER ===")
print(f"Duplicate transaction_ids: {ledger.duplicated('transaction_id').sum()}")
print(f"Null counts:\n{ledger.isnull().sum()}")

print("\n=== GATEWAY ===")
print(f"Duplicate transaction_ids: {gateway.duplicated('transaction_id').sum()}")
print(f"Null counts:\n{gateway.isnull().sum()}")


=== LEDGER ===
Duplicate transaction_ids: 0
Null counts:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64

=== GATEWAY ===
Duplicate transaction_ids: 0
Null counts:
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64


### 3.3 Identify Records Missing in Gateway

In [4]:
ledger_ids = set(ledger['transaction_id'])
gateway_ids = set(gateway['transaction_id'])

missing_in_gateway = ledger[~ledger['transaction_id'].isin(gateway_ids)].copy()
print(f"Records in ledger but NOT in gateway: {len(missing_in_gateway)}")
display(missing_in_gateway)

missing_in_gateway.to_csv(f"{OUT_DIR}/missing_in_gateway.csv", index=False)


Records in ledger but NOT in gateway: 2


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
3,R004,2026-03-02,M003,2100.0,success,Card
9,R010,2026-03-05,M004,2500.0,success,Wallet


### 3.4 Identify Records Missing in Ledger

In [5]:
missing_in_ledger = gateway[~gateway['transaction_id'].isin(ledger_ids)].copy()
print(f"Records in gateway but NOT in ledger: {len(missing_in_ledger)}")
display(missing_in_ledger)

missing_in_ledger.to_csv(f"{OUT_DIR}/missing_in_ledger.csv", index=False)


Records in gateway but NOT in ledger: 1


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
8,R011,2026-03-05,M003,1800.0,success,Card


### 3.5 Identify Amount Mismatches

In [6]:
common = pd.merge(ledger, gateway, on='transaction_id', suffixes=('_ledger', '_gateway'))

amount_mismatch = common[abs(common['amount_usd_ledger'] - common['amount_usd_gateway']) > 0.01].copy()
amount_mismatch = amount_mismatch[['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway']].copy()
amount_mismatch['difference'] = (amount_mismatch['amount_usd_gateway'] - amount_mismatch['amount_usd_ledger']).round(2)

print(f"Amount mismatches: {len(amount_mismatch)}")
display(amount_mismatch)

amount_mismatch.to_csv(f"{OUT_DIR}/amount_mismatches.csv", index=False)


Amount mismatches: 2


,transaction_id,amount_usd_ledger,amount_usd_gateway,difference
1,R002,850.0,900.0,50.0
6,R008,640.0,600.0,-40.0


### 3.6 Identify Status Mismatches

In [7]:
status_mismatch = common[common['status_ledger'] != common['status_gateway']].copy()
status_mismatch = status_mismatch[['transaction_id', 'status_ledger', 'status_gateway']].copy()

print(f"Status mismatches: {len(status_mismatch)}")
display(status_mismatch)

status_mismatch.to_csv(f"{OUT_DIR}/status_mismatches.csv", index=False)


Status mismatches: 1


,transaction_id,status_ledger,status_gateway
3,R005,success,failed


### 3.7 Build Final Reconciliation Report

In [8]:
recon = common.copy()
recon['amount_match'] = abs(recon['amount_usd_ledger'] - recon['amount_usd_gateway']) <= 0.01
recon['status_match'] = recon['status_ledger'] == recon['status_gateway']
recon['reconciled'] = recon['amount_match'] & recon['status_match']
recon['issue_type'] = recon.apply(
    lambda r: ('amount_mismatch ' if not r['amount_match'] else '') +
              ('status_mismatch' if not r['status_match'] else ''),
    axis=1
).str.strip()

report = recon[[
    'transaction_id', 'transaction_date_ledger', 'merchant_id_ledger',
    'amount_usd_ledger', 'amount_usd_gateway', 'status_ledger', 'status_gateway',
    'reconciled', 'issue_type'
]].rename(columns={
    'transaction_date_ledger': 'transaction_date',
    'merchant_id_ledger': 'merchant_id'
})

print(f"Total records compared: {len(report)}")
print(f"Fully reconciled: {report['reconciled'].sum()}")
print(f"Issues found: {(~report['reconciled']).sum()}")
display(report)

report.to_csv(f"{OUT_DIR}/reconciliation_report.csv", index=False)


Total records compared: 8
Fully reconciled: 5
Issues found: 3


,transaction_id,transaction_date,merchant_id,amount_usd_ledger,amount_usd_gateway,status_ledger,status_gateway,reconciled,issue_type
0,R001,2026-03-01,M001,1200.0,1200.0,success,success,True,
1,R002,2026-03-01,M002,850.0,900.0,success,success,False,amount_mismatch
2,R003,2026-03-02,M001,500.0,500.0,success,success,True,
3,R005,2026-03-03,M004,7200.0,7200.0,success,failed,False,status_mismatch
4,R006,2026-03-03,M002,950.0,950.0,success,success,True,
5,R007,2026-03-04,M005,3300.0,3300.0,failed,failed,True,
6,R008,2026-03-04,M001,640.0,600.0,success,success,False,amount_mismatch
7,R009,2026-03-05,M002,4100.0,4100.0,success,success,True,


### 3.8 Summary Metrics

In [9]:
ledger_total = ledger['amount_usd'].sum()
gateway_total = gateway['amount_usd'].sum()
amount_at_risk = (
    amount_mismatch['difference'].abs().sum() +
    missing_in_gateway['amount_usd'].sum() +
    missing_in_ledger['amount_usd'].sum()
)

summary_metrics = {
    "total_ledger_rows": int(len(ledger)),
    "total_gateway_rows": int(len(gateway)),
    "missing_in_gateway_count": int(len(missing_in_gateway)),
    "missing_in_ledger_count": int(len(missing_in_ledger)),
    "amount_mismatch_count": int(len(amount_mismatch)),
    "status_mismatch_count": int(len(status_mismatch)),
    "reconciliation_issue_count": int(len(missing_in_gateway) + len(missing_in_ledger) + len(amount_mismatch) + len(status_mismatch)),
    "ledger_total_amount": round(float(ledger_total), 2),
    "gateway_total_amount": round(float(gateway_total), 2),
    "amount_at_risk": round(float(amount_at_risk), 2)
}

print(json.dumps(summary_metrics, indent=2))

with open("../04_python/summary_metrics.json", 'w') as f:
    json.dump(summary_metrics, f, indent=2)


{
  "total_ledger_rows": 10,
  "total_gateway_rows": 9,
  "missing_in_gateway_count": 2,
  "missing_in_ledger_count": 1,
  "amount_mismatch_count": 2,
  "status_mismatch_count": 1,
  "reconciliation_issue_count": 6,
  "ledger_total_amount": 23340.0,
  "gateway_total_amount": 20550.0,
  "amount_at_risk": 6490.0
}


## Part 4: JSON Normalization

In [10]:
with open(f"{DATA_DIR}/api_response_sample.json") as f:
    api_data = json.load(f)

print(f"Source: {api_data['source']}")
print(f"Generated at: {api_data['generated_at']}")
print(f"Batches: {len(api_data['batches'])}")


Source: QuickPay Settlement API
Generated at: 2026-03-07T10:00:00Z
Batches: 2


### 4.1 Flatten Nested JSON to Tabular Form

In [11]:
rows = []
for batch in api_data['batches']:
    for s in batch['settlements']:
        rows.append({
            'batch_id': batch['batch_id'],
            'merchant_id': batch['merchant']['merchant_id'],
            'merchant_name': batch['merchant']['merchant_name'],
            'merchant_region': batch['merchant']['region'],
            'settlement_id': s['settlement_id'],
            'amount_usd': s['amount_usd'],
            'status': s['status'],
            'processed_at': s['processed_at'],
            'bank_name': s['bank']['name'],
            'bank_country': s['bank']['country'],
            'generated_at': api_data['generated_at'],
            'source': api_data['source'],
        })

api_norm = pd.DataFrame(rows)
print(f"Normalized shape: {api_norm.shape}")
display(api_norm)


Normalized shape: (6, 12)


,batch_id,merchant_id,merchant_name,merchant_region,settlement_id,amount_usd,status,processed_at,bank_name,bank_country,generated_at,source
0,B001,M001,Alpha Mart,APAC,S001,1520.5,settled,2026-03-07T08:10:00Z,Bank A,IN,2026-03-07T10:00:00Z,QuickPay Settlement API
1,B001,M001,Alpha Mart,APAC,S002,980.0,pending,2026-03-07T08:45:00Z,Bank A,IN,2026-03-07T10:00:00Z,QuickPay Settlement API
2,B001,M001,Alpha Mart,APAC,S003,640.0,settled,2026-03-07T09:15:00Z,Bank B,SG,2026-03-07T10:00:00Z,QuickPay Settlement API
3,B002,M004,Delta Travels,US,S004,2100.0,settled,2026-03-07T08:20:00Z,Bank C,US,2026-03-07T10:00:00Z,QuickPay Settlement API
4,B002,M004,Delta Travels,US,S005,500.0,failed,2026-03-07T08:50:00Z,Bank C,US,2026-03-07T10:00:00Z,QuickPay Settlement API
5,B002,M004,Delta Travels,US,S006,7200.0,settled,2026-03-07T09:30:00Z,Bank C,US,2026-03-07T10:00:00Z,QuickPay Settlement API


### 4.2 Clean Column Names and Convert Datetime Fields

In [12]:
# Column names already clean (snake_case)
api_norm['processed_at'] = pd.to_datetime(api_norm['processed_at'])
api_norm['generated_at'] = pd.to_datetime(api_norm['generated_at'])

print("dtypes:")
print(api_norm.dtypes)
print("\nSettlement status breakdown:")
print(api_norm['status'].value_counts())
print(f"\nTotal settled amount: ${api_norm[api_norm['status']=='settled']['amount_usd'].sum():,.2f}")


dtypes:
batch_id                           str
merchant_id                        str
merchant_name                      str
merchant_region                    str
settlement_id                      str
amount_usd                     float64
status                             str
processed_at       datetime64[us, UTC]
bank_name                          str
bank_country                       str
generated_at       datetime64[us, UTC]
source                             str
dtype: object

Settlement status breakdown:
status
settled    4
pending    1
failed     1
Name: count, dtype: int64

Total settled amount: $11,460.50


### 4.3 Save Normalized Output

In [13]:
api_norm.to_csv(f"{OUT_DIR}/api_normalized.csv", index=False)
print(f"Saved api_normalized.csv with {len(api_norm)} rows")
display(api_norm)


Saved api_normalized.csv with 6 rows


,batch_id,merchant_id,merchant_name,merchant_region,settlement_id,amount_usd,status,processed_at,bank_name,bank_country,generated_at,source
0,B001,M001,Alpha Mart,APAC,S001,1520.5,settled,2026-03-07 08:10:00+00:00,Bank A,IN,2026-03-07 10:00:00+00:00,QuickPay Settlement API
1,B001,M001,Alpha Mart,APAC,S002,980.0,pending,2026-03-07 08:45:00+00:00,Bank A,IN,2026-03-07 10:00:00+00:00,QuickPay Settlement API
2,B001,M001,Alpha Mart,APAC,S003,640.0,settled,2026-03-07 09:15:00+00:00,Bank B,SG,2026-03-07 10:00:00+00:00,QuickPay Settlement API
3,B002,M004,Delta Travels,US,S004,2100.0,settled,2026-03-07 08:20:00+00:00,Bank C,US,2026-03-07 10:00:00+00:00,QuickPay Settlement API
4,B002,M004,Delta Travels,US,S005,500.0,failed,2026-03-07 08:50:00+00:00,Bank C,US,2026-03-07 10:00:00+00:00,QuickPay Settlement API
5,B002,M004,Delta Travels,US,S006,7200.0,settled,2026-03-07 09:30:00+00:00,Bank C,US,2026-03-07 10:00:00+00:00,QuickPay Settlement API
